# Module 5: Giving the agent knowledge it can search

The agent answers a maintenance question by guessing. You give it a search tool over Kestrel's service bulletins, check its answer against the passages the search found, and see what it does when the bulletins don't cover a question.

In [ ]:
# Load the lab's notebook helpers, then work from the agent/ folder.
import labnb as lab
%cd ../agent

## Start from module 5

This cell puts the agent back where module 5 starts: the starter code plus the fixes from modules 2
to 4. It copies any edits you made to `agent/.reset-backup/` first, so nothing you wrote is lost.

In [ ]:
# Put the agent back in module 5's starting state.
!python reset.py 5

## 1. An answer the model can't know

Fault F-21 is described only in Kestrel's service bulletins, and no model was trained on them. So far
the agent's only tools are `find_part`, which looks up parts, and `check_stock`, which checks stock at
a depot.

In [ ]:
# Ask the agent the lab's question 'm5':
#   "A K-400 is showing fault F-21. What should the technician do?"
!python agent.py --q m5
# Show a table of this notebook's runs, one row each:
#   turns: how many times the agent called the model
#   ended by: what stopped the run
#   tools run: the tools the agent ran, and each result's category
#   answer: what the agent printed
lab.evidence(columns=("turns", "ended by", "tools run", "answer"))
# Show each tool the newest run called, with its arguments and exactly what it returned.
lab.tool_results()
# A link to the newest run's trace in MLflow.
lab.trace_link()

The answer is specific and sounds plausible, but the model made it up. The *tools run* column
shows `find_part (no_results)`, and the tool results under the table show what the model sent:
it looked up "F-21" as a part and got `no_results`, because a fault code isn't a part. Then it answered from its own training. No tool failed, so module
4's contract has nothing to catch. `no_results` from the parts catalogue says nothing about what F-21 means, and the
agent has no tool that does.

In the trace you'll see `model_call_1`, a `find_part` span for the parts lookup, and `model_call_2`,
which wrote the answer. There's no span for searching the service bulletins, because the agent can't
search them yet. After the fix you'll see one called `retrieve_bulletins`.

The bulletins are already loaded into a PostgreSQL database in your project. Each bulletin is split
into passages, one per section (*Applies to*, *Symptom*, *Action* and so on), and each passage is
stored with its `source`: the ID of the bulletin it came from, such as SB-2024-102. The `source` is
what lets you check an answer, because you can open the bulletin it names. A search returns up to
three passages, the ones closest to the search text. Each comes with a `score` from 0 to 1, where
higher means a closer match. Passages under 0.35 are dropped, and if none are left the search returns
`no_results`.

## 2. The knowledge it could use (stretch goal)

Skip this step if you're short of time. It searches the database directly, outside the agent, so you
can see what a search returns before the agent uses one.

The database uses pgvector, a PostgreSQL extension that stores each passage as a vector, a list of
numbers, and ranks passages by how close their vectors are to the search text's.
[`retrieval.py`](../agent/retrieval.py) is the lab's search code. It builds the vectors from the
passages' words. A real system would use a model trained to turn text into vectors, called an
embedding model. You don't need to read `retrieval.py` to continue.

In [ ]:
# Search the bulletins for "fault F-21" and print the passages that come back.
!python retrieval.py "fault F-21"

All three passages come from SB-2024-102. Its *Action* passage is the bulletin's fix: bleed air
from the coolant loop, then confirm the loop pressure reaches 2.1 bar. Look for it again in step 4.

## 3. Give the agent the search tool

You copy in module 5's versions of two files. [`tools.py`](../agent/tools.py), the tools the model is
offered, gets the new tool, `search_bulletins`. [`agent.py`](../agent/agent.py), the loop, gets new
instructions for using it.

In [ ]:
# Replace tools.py and agent.py with module 5's versions.
!cp solutions/module-05/tools.py solutions/module-05/agent.py .
# Show what changed in tools.py since module 4. Lines starting with + are new.
!git diff --no-index solutions/module-04/tools.py tools.py

Apart from its docstring and comments, `tools.py` gains three lines. It imports the tool's definition,
`SEARCH_BULLETINS_DEFINITION`, and its function, `search_bulletins`, from
[`retrieval.py`](../agent/retrieval.py), the lab's search code. It adds the definition to
`TOOL_DEFINITIONS`, the list of tools the model is offered, and the function to `TOOL_FUNCTIONS`,
which the agent uses to run a tool by name. The comment on the new `TOOL_DEFINITIONS` line says the
tool is now registered, which here means only that it's in that list.

In [ ]:
# Show the search tool's definition, the part the model reads.
lab.show_code("retrieval.py", "SEARCH_BULLETINS_DEFINITION")

The model reads this definition to decide when to search and what to send. It writes `query`
itself, and it can add `product` to search only one chiller model's bulletins. The tool returns
module 4's contract, with the passages under `data`.

The new instructions aren't in `config.toml`. `run_agent` in `agent.py` adds them to the system
prompt it reads from `config.toml`.

In [ ]:
# Show what changed in agent.py since module 4. Lines starting with + are new, - removed.
!git diff --no-index solutions/module-04/agent.py agent.py

The first five added lines are rules about part numbers and failed checks. They replace the
instruction module 4's agent added only after a failed check, which is the block removed further down.
The rest is new. The agent must cite each bulletin ID next to the claim it supports, end its answer
with a "Retrieved sources:" line listing every source the search returned, and, if the search returns
`no_results`, say the bulletins don't cover the question instead of inventing an answer.

```mermaid
flowchart LR
  Q["The F-21 question"] --> M["Model"]
  M -- "search_bulletins: a query the model writes" --> T["Retrieval tool"]
  T --> DB[("pgvector: Kestrel service bulletins")]
  DB --> T
  T -- "passages with their source ids, in module 4's contract" --> M
  M --> ANS["An answer that lists its sources"]
```

## 4. Ask again

Same question as step 1, but now the agent can search the bulletins.

In [ ]:
# Ask the agent the lab's question 'm5':
#   "A K-400 is showing fault F-21. What should the technician do?"
!python agent.py --q m5
# Show each tool the newest run called, with its arguments and exactly what it returned.
lab.tool_results()
# A link to the newest run's trace in MLflow.
lab.trace_link()

Start with the arguments. The model writes its own search, and the words it chooses and the product
filter decide what comes back. In the lab's test runs it sent `"query": "F-21"` with
`"product": "K-400"` and got two passages: the *Symptom* passage from SB-2024-102, and the *Applies
to* passage from SB-2024-117, a bulletin about servicing the K-400's coolant pump seal. SB-2024-102's *Action*
passage, the one with the fix, isn't there. If you did step 2, compare: the search for "fault F-21"
returned it.

Open the trace with the link above. Under the `search_bulletins` tool span there's now a
`retrieve_bulletins` span. Its outputs list the same passages, each labelled with its bulletin ID.

**This is the module's success signal:** `search_bulletins` was called, a passage from SB-2024-102
came back, the answer names the fault as that passage does ("coolant loop pressure low"), and it lists
its sources.

**Your read:** put the answer next to the passages. Which parts of the answer come from a passage, and
which don't?

<details><summary>Check your answer</summary>

The fault's name, "coolant loop pressure low", comes from SB-2024-102's *Symptom* passage. The pump
assembly KI-CP-40817 and seal kit KI-SK-40233 come from SB-2024-117's *Applies to* passage, which lists
the K-400s that seal bulletin covers and says nothing about F-21. Advice such as verifying coolant
levels comes from no passage at all.
</details>

The instructions ask for each bulletin ID next to the claim it supports. In the lab's test runs the
answer listed the IDs only at the end, so the answer alone doesn't tell you which claim came from
which bulletin. The model followed that instruction only in part, which is why you matched the claims
to the passages yourself.

So the answer is grounded: parts of it rest on passages the search returned, and it says which
bulletins they came from. It isn't complete, though. SB-2024-102's fix is to bleed air from the coolant
loop at the high point vent, confirm the pressure reaches 2.1 bar within 90 seconds, and inspect the
pump and its seal only if the fault returns. The search the model wrote never returned that passage,
so the answer leaves out bleeding the air and the 2.1 bar check, and its part numbers come from
SB-2024-117. You can read the whole bulletin in
[`SB-2024-102.md`](../agent/fixtures/bulletins/SB-2024-102.md). The answer lists its sources, so
you can check it against them. That check is how you found the missing bleed and pressure steps. You
don't fix the gap in this module.

## 5. Ask something the bulletins don't cover

Kestrel makes no K-900, so no bulletin covers one.

In [ ]:
# Ask the agent the lab's question 'm5-miss':
#   "What's the service interval on a Kestrel K-900?"
!python agent.py --q m5-miss
# Show each tool the newest run called, with its arguments and exactly what it returned.
lab.tool_results()
# Show a table of this notebook's runs (the last 3), one row each:
#   question: the question asked
#   tools run: the tools the agent ran, and each result's category
#   answer: what the agent printed
lab.evidence(columns=("question", "tools run", "answer"), last=3)

The search returns `no_results`, and the agent says the bulletins don't cover the question instead
of inventing an interval. Compare that with step 1, where the agent had nothing to search
and invented a fix for F-21.

Two things changed since step 1: the agent can now search, and its instructions tell it what to say
when the search comes back empty.

In the lab's test runs the model searched with `"product": "K-900"`.
No bulletin lists the K-900, so that filter leaves no passages to score, and the tool returns
`no_results`.

The answer ends with an empty "Retrieved sources:" line. It isn't cut off. The instructions say to
always end with that line, and this time there were no sources to list.

Nothing in the lab checks that a retrieved passage answers the question. Module 6's F-21 check asks
only that a passage from SB-2024-102 came back and that the answer names SB-2024-102, so step 4's
incomplete answer would pass it.

## If the result differs

If step 2's search, or the agent's `search_bulletins` call in step 4, comes back with
`upstream_unavailable` and "knowledge store unreachable", the agent can't reach the database. It may
still be starting, so wait a minute and run the cell again. If that doesn't help, open a terminal
(*File > New > Terminal*), run `cd ~/agent-antipatterns-lab/agent`, then `env | grep PG`, and check
that `PGHOST`, `PGPORT`, `PGDATABASE`, `PGUSER` and `PGPASSWORD` are all set. Your workbench sets
them when it starts. If one is missing, the database wasn't set up for your project, and you can't
fix that from the notebook. Ask whoever runs your lab session for help.

If step 2's search for "fault F-21" returns `no_results`, the bulletins aren't loaded. In the same
terminal and folder, run `python ingest_corpus.py`. [`ingest_corpus.py`](../agent/ingest_corpus.py)
loads the bulletins into the database, and it should print "ingested 12 passages from 5 bulletins".
Then run the cell again. If only the agent's search in step 4 returns `no_results`, the query it
wrote may be the cause. Run step 2's cell: if that finds passages, the bulletins are loaded, so run
step 4 again.

If step 4's tool results show `find_part` and no `search_bulletins`, the agent doesn't have the new
tool yet. Run step 3's first cell again, then step 4.

After the fix, you should see a passage from SB-2024-102 in step 4's tool results and SB-2024-102 in
the answer's sources. Your answer's wording may differ from the one described in step 4. Check it
against the passages the same way, because finding a passage doesn't prove the answer used it. If
your agent's query differs, so will its passages. If the *Action* passage comes back, the answer may
include the bleed and the 2.1 bar check. Do the same check with the passages your run got: which parts
of the answer come from them, and which don't?

For K-900, `search_bulletins` should return `no_results`, and the answer should say the bulletins
don't cover the question, with no made-up interval.